In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import csv

In [2]:
def plot_series(x, y, format="-", start=0, end=None,
                title=None, xlabel, ylabel):
    
    # Setup dimensions of the graph figure
    plt.figure(figsize=(10, 6))

    # Check if there are more than two series to plot
    if type(y) is tuple:

      # Loop over the y elements
      for y_curr in y:

        # Plot the x and current y values
        plt.plot(x[start:end], y_curr[start:end], format)

    else:
      # Plot the x and y values
      plt.plot(x[start:end], y[start:end], format)

    # Label the x-axis
    plt.xlabel(xlabel)

    # Label the y-axis
    plt.ylabel(ylabel)

    # Set the legend
    if legend:
        plt.legend(legend)

    # Set the title
    plt.title(title)

    # Overlay a grid on the graph
    plt.grid(True)

    # Draw the graph on screen
    plt.show()

SyntaxError: non-default argument follows default argument (1574045122.py, line 2)

In [ ]:
#initialize lists
Date= []
Open=[]
High=[]
Low=[]
Close=[]

with open('./AMZN.csv') as csvfile:

  # Initialize reader
  reader = csv.reader(csvfile, delimiter=',')

  # Skip the first line
  next(reader)

  # Append row and sunspot number to lists
  for row in reader:
    Date.append(int(row[0]))
    Open.append(float(row[2]))
    High.append(float(row[3]))
    Low.append(float(row[4]))
    Close.append(float(row[5]))

# Convert lists to numpy arrays
time = np.array(Date)
Open = np.array(Open)
High = np.array(High)
Low = np.array(Low)
Close= np.array(Close)


# plot_series(time, (Open, Close), xlabel='Day', ylabel='Daily Open Price')

In [ ]:
# Create subplots
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

# Plot Open prices
axs[0, 0].plot(time, Open, label='Open', color='blue')
axs[0, 0].set_title('Open Prices')
axs[0, 0].set_xlabel('Time')
axs[0, 0].set_ylabel('Price')
axs[0, 0].grid(True)

# Plot Close prices
axs[0, 1].plot(time, Close, label='Close', color='red')
axs[0, 1].set_title('Close Prices')
axs[0, 1].set_xlabel('Time')
axs[0, 1].set_ylabel('Price')
axs[0, 1].grid(True)

# Plot High prices
axs[1, 0].plot(time, High, label='High', color='green')
axs[1, 0].set_title('High Prices')
axs[1, 0].set_xlabel('Time')
axs[1, 0].set_ylabel('Price')
axs[1, 0].grid(True)

# Plot Low prices
axs[1, 1].plot(time, Low, label='Low', color='purple')
axs[1, 1].set_title('Low Prices')
axs[1, 1].set_xlabel('Time')
axs[1, 1].set_ylabel('Price')
axs[1, 1].grid(True)

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Define the split time
split_time = 6500

# Get the train set
time_train = time[:split_time]
x_train = Open[:split_time]

# Get the validation set
time_valid = time[split_time:]
x_valid = Open[split_time:]

In [ ]:
def windowed_dataset(series, window_size, batch_size, shuffle_buffer):

    # Generate a TF Dataset from the series values
    dataset = tf.data.Dataset.from_tensor_slices(series)

    # Window the data but only take those with the specified size
    dataset = dataset.window(window_size + 1, shift=1, drop_remainder=True)

    # Flatten the windows by putting its elements in a single batch
    dataset = dataset.flat_map(lambda window: window.batch(window_size + 1))

    # Create tuples with features and labels
    dataset = dataset.map(lambda window: (window[:-1], window[-1]))

    # Shuffle the windows
    dataset = dataset.shuffle(shuffle_buffer)

    # Create batches of windows
    dataset = dataset.batch(batch_size).prefetch(1)
    
#     for x,y in dataset:
#         print("x = ", x.numpy())
#         print("y = ", y.numpy())
#         print()
    return dataset

In [ ]:
# Parameters
window_size = 60
batch_size = 32
shuffle_buffer_size = 1000

# Generate the dataset windows
train_set = windowed_dataset(x_train, window_size, batch_size, shuffle_buffer_size)

In [ ]:
# Build the Model
model = tf.keras.models.Sequential([
  tf.keras.layers.Conv1D(filters=64, kernel_size=3,
                      strides=1,
                      activation="relu",
                      padding='causal',
                      input_shape=[window_size, 1]),
  tf.keras.layers.GRU(units=50, return_sequences=True, input_shape=(window_size, 1)),
  tf.keras.layers.GRU(units=50),
  tf.keras.layers.Dense(30, activation="relu"),
  tf.keras.layers.Dense(10, activation="relu"),
  tf.keras.layers.Dense(1),
  tf.keras.layers.Lambda(lambda x: x * 400)
])

 # Print the model summary
model.summary()

In [ ]:
# Get initial weights
init_weights = model.get_weights()

# Set the learning rate
learning_rate = 2e-6

# Set the optimizer
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

# Set the training parameters
model.compile(loss=tf.keras.losses.Huber(),
              optimizer=optimizer,
              metrics=["mae"])